<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: People in images

**[People in images](https://learning.nextia-ai.com/courses/vision/m08/people-in-images/)** · Vision AI: See, Count and Inspect · Module 8, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find faces in the background of a photo, blur them before any copy is kept, check that the blur works, and choose the face finder's threshold from the cost of each mistake.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (the returns and catalogue photos and the face finder, 31 MB). No account and no key. |
| **You should know** | The review folder, from [The image service and its review queue](https://learning.nextia-ai.com/courses/vision/m08/the-image-service-and-its-review-queue/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 and OpenCV 5.0.0 (opencv-python-headless), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/people-in-images/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M08-L04-people-in-images/people-in-images-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, opencv-python-headless 5.0.0.93, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 opencv-python-headless==5.0.0.93
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import cv2; print("opencv-python-headless", cv2.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 31 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "compare.py": "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3",
    "models/faces/face_detection_yunet_2023mar.onnx": "8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4",
    "models/faces/LICENSE": "c83b8120c50ccbd4c4f96edf53141bdd566ebb8f8e9227e415326aa1b1aba958",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Find the faces in the month

The month has 135 returns photos. In three of them, our code pasted a face from a public-domain painting into the background, over a candle: the case of a person who leans over the table. No photo in this course shows a real person.

The service's face finder is YuNet (OpenCV Zoo, MIT licence, 230 KB). It gives each face a box and a score from 0 to 1; the service keeps faces from 0.6.

In [ ]:
import cv2
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)  # OpenCV 5 prints a harmless warning otherwise

def face_finder(threshold=0.6):
    """YuNet: returns a function photo -> list of faces [x, y, w, h, score]."""
    det = cv2.FaceDetectorYN.create("models/faces/face_detection_yunet_2023mar.onnx", "", (320, 320), threshold)
    def find(img):
        a = cv2.cvtColor(np.asarray(img.convert("RGB")), cv2.COLOR_RGB2BGR)
        det.setInputSize((a.shape[1], a.shape[0]))
        _, faces = det.detect(a)
        return [] if faces is None else [[int(v) for v in f[:4]] + [round(float(f[-1]), 3)] for f in faces]
    return find

def flagged(find, files):
    """The files where the finder sees at least one face."""
    return [f for f in files if find(Image.open("data/" + f))]

month = pd.read_csv("data/returns/month_manifest.csv")
catalogue = pd.read_csv("data/catalogue/manifest.csv")

find = face_finder(0.6)
with_faces = flagged(find, month["file"])
print(len(with_faces), "photos with a face:", with_faces)
show(["data/" + f for f in with_faces], [f.split("/")[-1] for f in with_faces], size=3)

> **Check.** 3 photos: R-fcd0915d (Vermeer), R-fd29f7cf (van Gogh) and R-e382ce16 (Leonardo).

## 2. Blur every face before a copy is kept

The blur makes the face region tiny (12 times smaller), makes it big again, and smooths it. A margin of 30% around the box also covers hair and ears.

In [ ]:
from PIL import ImageFilter

def blur(img, boxes, margin=0.3):
    out = img.convert("RGB").copy()
    for x, y, w, h, *_ in boxes:
        m = int(margin * max(w, h))
        x0, y0, x1, y1 = max(0, x - m), max(0, y - m), min(out.width, x + w + m), min(out.height, y + h + m)
        region = out.crop((x0, y0, x1, y1))
        small = region.resize((max(1, region.width // 12), max(1, region.height // 12)), Image.BILINEAR)
        out.paste(small.resize(region.size, Image.NEAREST).filter(ImageFilter.GaussianBlur(4)), (x0, y0))
    return out

Path("review").mkdir(exist_ok=True)
for f in with_faces:
    img = Image.open("data/" + f)
    blur(img, find(img)).save("review/" + f.split("/")[-1], quality=90)
show(["review/" + f.split("/")[-1] for f in with_faces], ["blurred"] * 3, size=3)

## 3. Check the blur

A test, not a look: run the face finder again on the blurred copies. It should find nothing. And the detail (the spread of the pixel values) in the face box should fall clearly: here, to about half.

In [ ]:
for f in with_faces:
    name = f.split("/")[-1]
    original, copy = Image.open("data/" + f), Image.open("review/" + name)
    x, y, w, h, _ = find(original)[0]
    detail_before = np.asarray(original.convert("L"))[y:y + h, x:x + w].std()
    detail_after = np.asarray(copy.convert("L"))[y:y + h, x:x + w].std()
    print(name, "faces found after the blur:", len(find(copy)), "| detail", round(detail_before, 1), "->", round(detail_after, 1))

## 4. The threshold: missed faces against false faces

The catalogue has no person: four filters and a look at every photo removed them. So every face that the finder sees in the catalogue is a false face: a pattern on a bowl, a reflection on a mug.

> **Predict.** At the service's threshold, 0.6, how many of the 733 catalogue photos get a false face?

In [ ]:
print("threshold 0.6:", len(flagged(find, catalogue["file"])), "catalogue photos with a false face")
strict = face_finder(0.9)
print("threshold 0.9:", len(flagged(strict, month["file"])), "of the 3 painted faces found")

> **Check.** 14 false faces at 0.6. At 0.9 there are none, but the finder misses one painted face (van Gogh scores 0.796). For blurring, a missed face is the expensive mistake; a blurred bowl costs almost nothing.

## 5. Your turn: a lower threshold

> **Your turn.** Put the number of catalogue photos with a "face" at threshold **0.3** in `false_03` (an integer). Then decide: would you use 0.3 in the service?

In [ ]:
false_03 = len(flagged(face_finder(0.3), catalogue["file"]))
print(false_03)

In [ ]:
expect('false_03', false_03, 83)

At 0.3 the finder flags many catalogue photos. In the service, each flag sends a photo to the review folder, so a person's minute is spent on a bowl. At 0.6 the month's three faces are all found and only a few catalogue photos are flagged. Choose the threshold with the price of each mistake, as Hamid chose his lines.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/people-in-images/) explains what to keep, what to delete, and which person-related tasks this course does not build, with the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Visual inspection with few defects](https://learning.nextia-ai.com/courses/vision/m09/visual-inspection-with-few-defects/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m08/people-in-images/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code. The only faces are three public-domain paintings, added by our code; no photo shows a real person.